# SAAM Project 2026
---
## Part I : Standard Portfolio Allocation
### Group N : Emerging Markets / Scope 1
---

## 1. Project Overview

This notebook implements Part I of the SAAM project for Group N, which focuses on firms from Emerging Markets and uses Scope 1 emissions as the climate metric for the broader project.  

The objective of Part I is to construct and compare two portfolio strategies over the out-of-sample period 2014–2025:

- a long-only minimum variance portfolio,
- a value-weighted benchmark portfolio.

The notebook follows the project instructions by:
- cleaning and preparing the financial data,
- constructing monthly stock returns,
- defining a dynamic investment universe,
- estimating portfolio weights annually using a rolling 120-month window,
- backtesting the portfolios on a monthly basis,
- and comparing their performance using standard portfolio statistics.

---
[GUIDE ÉQUIPE – À SUPPRIMER AVANT RENDU]

Cette section sert à expliquer dès le début ce que couvre ce notebook.
Pour l’instant, ce notebook couvre uniquement la Part I :
- minimum variance portfolio
- value-weighted benchmark
- comparaison des performances

Quand vous reprendrez le notebook pour la Part II, il faudra prolonger la logique ici, mais sans réécrire toute l’introduction.

---


## 2. Environment Setup and Project Configuration
This section prepares the computational environment used throughout the notebook.

It loads the required Python libraries, defines the project folder structure,
and sets the global parameters used in the portfolio construction process.

Organizing these elements at the beginning of the notebook ensures that the
analysis remains reproducible and easy to maintain.

----
[GUIDE ÉQUIPE – À SUPPRIMER AVANT RENDU]

Cette section doit rester stable.
Elle contient :
- les librairies
- les chemins du projet
- les paramètres globaux

Si quelqu’un modifie les noms de dossiers, les fichiers de données ou les hypothèses générales, cela doit être fait ici en priorité.

---

### 2.1 Importing required libraries

The following libraries are required to run the analysis.

Pandas and NumPy are used for data manipulation, Matplotlib is used for
visualization, and SciPy provides the optimization routines used to
construct the minimum variance portfolio.

In [ ]:
# Standard libraries
import os
import warnings

warnings.filterwarnings("ignore")

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt

# Optimization
from scipy.optimize import minimize

: 

### 2.2 Verifying the Python environment

Before running the rest of the analysis, we verify that the required
libraries are correctly installed and accessible.

In [ ]:
# Display library versions to ensure reproducibility of the computational environment

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Imports loaded successfully.")

### 2.3 Display configuration

The following settings improve the readability of data tables displayed
in the notebook by controlling the number of rows, columns, and numerical formatting.

In [ ]:
# Configure pandas display settings to improve readability of DataFrame outputs

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 20)
pd.set_option("display.float_format", "{:.4f}".format)

### 2.4 Defining project directories

To keep the workflow organized and reproducible, the main project
directories are defined here. All file paths used later in the notebook
are constructed relative to the project base directory.

In [ ]:
from pathlib import Path

# Detect project root by searching for the folder that contains "data_raw"
current_dir = Path.cwd()

BASE_DIR = None
for parent in [current_dir] + list(current_dir.parents):
    if (parent / "data_raw").exists():
        BASE_DIR = parent
        break

if BASE_DIR is None:
    raise FileNotFoundError(
        "Could not locate the project folder containing 'data_raw'. "
        "Please ensure the notebook is somewhere inside the project directory."
    )

# Main folders
DATA_RAW = BASE_DIR / "data_raw"
DATA_PROCESSED = BASE_DIR / "data_processed"
OUTPUTS = BASE_DIR / "outputs"

# Output subfolders
TABLES_DIR = OUTPUTS / "tables"
FIGURES_DIR = OUTPUTS / "figures"
INTERMEDIATE_DIR = OUTPUTS / "intermediate"

print("Project root detected at:", BASE_DIR)

In [ ]:
# Verify that the project directories are correctly defined

print("Current working directory:", Path.cwd())
print("BASE_DIR:", BASE_DIR)
print("DATA_RAW:", DATA_RAW)
print("DATA_PROCESSED:", DATA_PROCESSED)
print("OUTPUTS:", OUTPUTS)

### 2.5 Creating required folders

The following step ensures that all output directories exist before
running the analysis. If a folder does not exist, it is automatically created.

In [ ]:
# Ensure that all required output directories exist.
# If a folder does not exist, it is created automatically.

for folder in [DATA_PROCESSED, OUTPUTS, TABLES_DIR, FIGURES_DIR, INTERMEDIATE_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Output directories checked and created if necessary.")

### 2.6 Global project parameters

This block defines the main parameters used throughout the project,
including the estimation window, filtering rules, and portfolio assumptions.
Centralizing these parameters simplifies future modifications.

In [ ]:
# Global project parameters
REGION = "EM"
CARBON_SCOPE = "SCOPE_1"

# Project timeline
START_ESTIMATION_YEAR = 2013
END_REBALANCE_YEAR = 2024
BACKTEST_START_YEAR = 2014
BACKTEST_END_YEAR = 2025

# Estimation and filtering rules
ROLLING_WINDOW_MONTHS = 120
MIN_HISTORY_MONTHS = 36
STALE_RETURN_THRESHOLD = 0.50
LOW_PRICE_THRESHOLD = 0.5

# Portfolio assumptions
INITIAL_WEALTH = 1_000_000
CARBON_REDUCTION_TARGET = 0.50
NET_ZERO_REDUCTION = 0.10

In [ ]:
# Display the main project parameters used throughout the notebook

print("Assigned region:", REGION)
print("Carbon scope:", CARBON_SCOPE)
print("Estimation window (months):", ROLLING_WINDOW_MONTHS)
print("Initial portfolio wealth (USD):", INITIAL_WEALTH)

## 3. Data Loading
This section loads the raw datasets used in Part I of the project.

At this stage, the objective is only to import the original files and
perform a minimal structural inspection. No financial cleaning or portfolio
construction is performed yet.

---

[GUIDE ÉQUIPE – À SUPPRIMER AVANT RENDU]

Cette section sert uniquement à charger les fichiers bruts et à vérifier leur structure.
On ne fait pas encore ici :
- le filtrage Emerging Markets
- le nettoyage des prix
- le calcul des rendements
- la création de l’univers dynamique

Tout cela ira dans la section 4.

----

### 3.1 Loading the raw datasets

The following datasets are loaded from the project folder.
For Part I, the key files are the static firm information, monthly return index data,
and monthly market capitalization data.

Annual carbon emissions are also loaded at this stage, even though the carbon objective is introduced in Part II. This is necessary because the project instructions require the investment universe in Part I to exclude firms without carbon data available at the end of each year.

In [ ]:
# Load the main datasets used in Part I:
# - Static firm information (region, identifiers, characteristics)
# - Monthly total return index prices
# - Monthly market capitalizations
# - Annual carbon emissions for the assigned scope

static = pd.read_excel(DATA_RAW / "Static_2025.xlsx")
prices_raw = pd.read_excel(DATA_RAW / "DS_RI_T_USD_M_2025.xlsx")
market_caps_raw = pd.read_excel(DATA_RAW / "DS_MV_T_USD_M_2025.xlsx")
carbon_raw = pd.read_excel(DATA_RAW / "DS_CO2_SCOPE_1_Y_2025.xlsx")

### 3.2 Inspecting the raw dataset structure

Before cleaning the data, we verify that the files have been loaded correctly
and contain the expected dimensions and key identifier columns.

In [ ]:
# Inspect dataset dimensions to verify that the files were loaded correctly

print("Static shape:", static.shape)
print("Prices raw shape:", prices_raw.shape)
print("Market caps raw shape:", market_caps_raw.shape)
print("Carbon raw shape:", carbon_raw.shape)

In [ ]:
# Inspect the structure of the static dataset (firm characteristics)

static.head()

In [ ]:
# Inspect the raw monthly price dataset to verify its structure,
# including identifier columns and potential Datastream extraction issues

prices_raw.head()

In [ ]:
# Inspect the raw market capitalization dataset to verify its structure
# and ensure consistency with the price dataset

market_caps_raw.head()

### 3.3 Checking identifier columns

The Datastream files contain both identifier columns and time-series columns.
At this stage, we verify that ISIN is available and that the first columns
match the expected raw file structure.

In [ ]:
# Inspect column names to verify dataset structure
# This helps identify identifier columns and date columns in the raw datasets

print("Static columns:", static.columns.tolist())
print("Prices raw first columns:", prices_raw.columns[:10].tolist())
print("Market caps raw first columns:", market_caps_raw.columns[:10].tolist())

In [ ]:
# Verify that the identifier column (ISIN) is present in the raw datasets

print("ISIN column in prices dataset:", "ISIN" in prices_raw.columns)
print("ISIN column in market caps dataset:", "ISIN" in market_caps_raw.columns)

### 3.4 Setting ISIN as the raw index

To facilitate later filtering and alignment across datasets, the ISIN code
is used as the row index for the raw price and market capitalization tables.
At this stage, non-time-series columns are still kept unchanged.

In [ ]:
# Set ISIN as the index to structure datasets as asset-by-time matrices

prices_raw = prices_raw.set_index("ISIN")
market_caps_raw = market_caps_raw.set_index("ISIN")

In [ ]:
# Confirm that ISIN has been correctly set as the index for both datasets

print("Prices raw index name:", prices_raw.index.name)
print("Market caps raw index name:", market_caps_raw.index.name)

In [ ]:
# Inspect the dataset after setting ISIN as the index
# This confirms the new row structure and helps identify remaining non-time-series columns

prices_raw.head()

## 4. Data Cleaning and Preparation

This section prepares the raw datasets for the empirical analysis.

The main objective is to isolate the firms belonging to the relevant investment region,
remove non-usable raw content from the Datastream files, and construct clean
time-series tables for prices and market capitalizations.

---
[GUIDE ÉQUIPE – À SUPPRIMER AVANT RENDU]

Cette section sert à transformer les fichiers bruts en tables exploitables.

À ce stade, on va seulement :
- garder les entreprises EM
- retirer les lignes parasites Datastream
- enlever la colonne NAME
- convertir les colonnes de dates
- préparer les tables finales prices / market caps

On ne calcule pas encore les rendements ici.

----

### 4.1 Filtering the Emerging Markets universe


The project is conducted for Group N, which is assigned to the Emerging Markets region.

The first cleaning step therefore consists in restricting the firm universe
to companies classified as Emerging Markets in the static reference table.

---

[GUIDE ÉQUIPE – À SUPPRIMER AVANT RENDU]

Ici on construit l’univers régional de départ.
Pour notre groupe, on garde uniquement les entreprises avec `Region = EM`.

Attention :
ce n’est pas encore l’univers investissable final.
C’est seulement le filtrage régional brut.
L’univers dynamique viendra plus tard.

---

In [ ]:
# Define the investment universe by selecting firms belonging to the assigned region

em_firms = static[static["Region"] == REGION].copy()
em_isins = em_firms["ISIN"].tolist()

print("Number of firms in the Emerging Markets universe:", len(em_isins))

In [ ]:
# Inspect the selected regional universe after filtering the static dataset

em_firms.head()

In [ ]:
# Prepare carbon emissions dataset

# If ISIN is still a column, use it as index
if "ISIN" in carbon_raw.columns:
    carbon_raw = carbon_raw.set_index("ISIN")

# Remove rows without valid identifiers
carbon_raw = carbon_raw[carbon_raw.index.notna()]

# Remove the non-time-series column (NAME) if still present
if "NAME" in carbon_raw.columns:
    carbon_clean = carbon_raw.drop(columns="NAME").copy()
else:
    carbon_clean = carbon_raw.copy()

# Keep only firms belonging to the EM universe
carbon_data = carbon_clean.loc[carbon_clean.index.isin(em_isins)].copy()

# Convert carbon values to numeric and coerce Datastream error strings to NaN
carbon_data = carbon_data.apply(pd.to_numeric, errors="coerce")

# Fill missing annual values forward as suggested in the project instructions
carbon_data = carbon_data.ffill(axis=1)

# Convert annual columns to end-of-year datetime
carbon_data.columns = pd.to_datetime(carbon_data.columns.astype(str), format="%Y")
carbon_data.columns = carbon_data.columns + pd.offsets.YearEnd(0)

# Ensure chronological order
carbon_data = carbon_data.sort_index(axis=1)

print("Carbon dataset shape:", carbon_data.shape)
carbon_data.head()

### 4.2 Removing invalid raw rows

Some Datastream exports may contain invalid rows generated during the data extraction process.

These rows typically appear with missing identifiers or error messages
in the first observation row. Such entries do not correspond to real firms
and must therefore be removed before proceeding with the data preparation.

---

[GUIDE ÉQUIPE – À SUPPRIMER AVANT RENDU]

Dans les fichiers Datastream, il arrive qu'une ligne d'erreur apparaisse
en haut du fichier (ex: "INVALID CODE OR EXPRESSION ENTERED").

Cette ligne n'est pas une entreprise et doit être supprimée.
On supprime simplement les lignes dont l'ISIN est manquant.

---

In [ ]:
# Remove rows with missing ISIN identifiers
# These rows may appear in Datastream exports due to extraction errors

prices_raw = prices_raw[prices_raw.index.notna()]
market_caps_raw = market_caps_raw[market_caps_raw.index.notna()]

In [ ]:
# Check dataset dimensions after removing rows with missing ISIN identifiers

print("Prices raw shape after cleaning:", prices_raw.shape)
print("Market caps raw shape after cleaning:", market_caps_raw.shape)

In [ ]:
# Inspect the price dataset after removing invalid rows

prices_raw.head()

### 4.3 Removing non-time-series columns

The Datastream price and market capitalization files contain descriptive columns
in addition to the monthly time-series data.

Since the portfolio construction relies exclusively on time-series information,
these descriptive columns must be removed to obtain clean numerical matrices.


---

[GUIDE ÉQUIPE – À SUPPRIMER AVANT RENDU]

Les fichiers Datastream contiennent encore la colonne `NAME`.
Elle sert uniquement à identifier visuellement les entreprises.

Pour les calculs financiers (rendements, covariance, optimisation),
on ne doit garder que les colonnes de dates.

Attention :
ISIN est déjà dans l'index, donc on ne supprime qu'une seule colonne ici.

---



In [ ]:
# Remove the non-time-series column (NAME) and keep only date columns
# This converts the dataset into an asset-by-time matrix

prices_clean = prices_raw.iloc[:, 1:].copy()
market_caps_clean = market_caps_raw.iloc[:, 1:].copy()

In [ ]:
# Check dataset dimensions after removing non-time-series columns

print("Prices clean shape:", prices_clean.shape)
print("Market caps clean shape:", market_caps_clean.shape)

In [ ]:
# Inspect the cleaned price matrix after removing non-time-series columns

prices_clean.head()

### 4.4 Converting time-series columns to datetime

To ensure proper time-series handling, the column labels representing
monthly observations are explicitly converted to datetime objects.

This allows chronological sorting and facilitates the later
construction of return series and portfolio backtests.


---

[GUIDE ÉQUIPE – À SUPPRIMER AVANT RENDU]

Même si les colonnes ressemblent déjà à des dates,
on force la conversion en datetime pour éviter tout problème
dans les manipulations temporelles.

C'est une bonne pratique standard en analyse financière.

---



In [ ]:
# Convert column labels to datetime format for time-series operations

prices_clean.columns = pd.to_datetime(prices_clean.columns)
market_caps_clean.columns = pd.to_datetime(market_caps_clean.columns)

In [ ]:
# Ensure columns (dates) are ordered chronologically

prices_clean = prices_clean.sort_index(axis=1)
market_caps_clean = market_caps_clean.sort_index(axis=1)

In [ ]:
# Verify that the columns are now a datetime index and inspect the first dates

print("Column index type:", type(prices_clean.columns))
print("First dates in price dataset:", prices_clean.columns[:5])

### 4.5 Preparing the final clean matrices

After cleaning the raw Datastream exports, the final step consists in
restricting the datasets to the relevant investment universe.

Only firms belonging to the Emerging Markets region are retained.
The resulting price and market capitalization matrices will serve
as the basis for the subsequent return calculations and portfolio construction.


---

[GUIDE ÉQUIPE – À SUPPRIMER AVANT RENDU]

Ici on applique réellement le filtre régional.

Jusqu'ici les matrices contiennent toutes les entreprises.
On garde uniquement celles présentes dans `em_isins`.

Les matrices finales utilisées dans tout le projet seront :

price_data
market_caps_data

---



In [ ]:
# Restrict datasets to the Emerging Markets investment universe

price_data = prices_clean.loc[prices_clean.index.isin(em_isins)].copy()
market_caps_data = market_caps_clean.loc[market_caps_clean.index.isin(em_isins)].copy()

In [ ]:
# Verify the dimensions of the EM investment universe matrices

print("Price matrix shape:", price_data.shape)
print("Market cap matrix shape:", market_caps_data.shape)

In [ ]:
# Inspect the final monthly price matrix for the Emerging Markets investment universe

price_data.head()

## 5. Construction of Monthly Returns

### 5.1 Filtering unrealistic prices

Datastream price series may contain observations corresponding to extremely
low prices, which often reflect illiquid securities, penny stocks,
or data extraction issues.

To mitigate the impact of such observations on the return calculations,
prices below 0.5 USD are treated as missing values.


---

[GUIDE ÉQUIPE – À SUPPRIMER AVANT RENDU]

On applique ici la règle utilisée dans l'ancien notebook.

Les prix < 0.5 sont considérés comme non fiables :
- penny stocks
- erreurs Datastream
- problèmes de liquidité

On remplace ces valeurs par NaN avant de calculer les rendements.

---



In [ ]:
# Ensure all price observations are numeric
price_data = price_data.apply(pd.to_numeric, errors="coerce")

# Identify abnormal price observations (Datastream convention: prices < 0.5)
(price_data < 0.5).sum().sum()

In [ ]:
# Replace abnormal Datastream price observations (< 0.5) with missing values

price_data[price_data < 0.5] = np.nan

In [ ]:
# ensure chronological order of price observations
price_data = price_data.sort_index(axis=1)

### 5.2 Computing monthly returns

Monthly returns are computed from the Datastream return index series using simple percentage changes between consecutive monthly observations.

Special attention is paid to firms that disappear at the end of the sample. In line with the project instructions, if a firm has a last valid price observation followed only by missing values, the first missing month is interpreted as a delisting event and assigned a return of −100%.

---
[GUIDE ÉQUIPE – À SUPPRIMER AVANT RENDU]

On garde ici la logique demandée par le projet :
- rendements simples mensuels
- pas de clipping
- si une firme disparaît définitivement en fin de série, on attribue un rendement de -100% au premier mois manquant après le dernier prix observé

---

----
Contenu à effacer mais important à voir en groupe:



In [ ]:
# Compute simple monthly returns from price indices
# fill_method=None avoids implicit filling around missing values

returns = price_data.pct_change(axis=1, fill_method=None)

# Approximate delisting returns:
# if a firm has a last valid price and all following prices are missing,
# assign -100% to the first missing month after the last valid observation

for isin in price_data.index:
    prices = price_data.loc[isin]
    valid_mask = prices.notna()

    if valid_mask.any():
        last_valid_pos = np.where(valid_mask.values)[0][-1]

        if last_valid_pos < len(prices.index) - 1:
            trailing_block = prices.iloc[last_valid_pos + 1:]

            if trailing_block.isna().all():
                first_missing_date = prices.index[last_valid_pos + 1]
                returns.loc[isin, first_missing_date] = -1.0

In [ ]:
# Diagnostic: inspect the most extreme monthly returns

stacked_returns = returns.stack()

print("Minimum return:", stacked_returns.min())
print("Maximum return:", stacked_returns.max())

print("\nFive most negative returns:")
print(stacked_returns.nsmallest(5))

print("\nFive most positive returns:")
print(stacked_returns.nlargest(5))

In [ ]:
stacked_returns = returns.stack()
print("Returns exactly -100%:", (stacked_returns == -1).sum())

In [ ]:
# Count very large return observations

print("Returns < -100%:", (stacked_returns < -1).sum())
print("Returns exactly -100%:", (stacked_returns == -1).sum())
print("Returns > 100%:", (stacked_returns > 1).sum())
print("Returns > 200%:", (stacked_returns > 2).sum())
print("Returns > 500%:", (stacked_returns > 5).sum())


The distribution of monthly returns was inspected to detect potential data anomalies. While a small number of extremely large positive returns were observed (above 200%), these observations correspond to firms with very small market capitalizations and therefore have negligible impact in the value-weighted benchmark. For consistency with the project instructions, returns were not winsorized.

----

In [ ]:
# Inspect the first few monthly returns to verify the return calculation
returns.iloc[:, :5].head()

### 5.3 Inspecting the return distribution

At this stage, the return matrix is inspected to verify that the price cleaning procedure has removed the most problematic observations. In particular, treating prices below 0.5 as missing values helps prevent mechanically inflated returns caused by extremely low price levels.

---
[GUIDE ÉQUIPE – À SUPPRIMER AVANT RENDU]

Ici on ne winsorise pas les rendements.
On se contente d’inspecter leur distribution après le nettoyage des prix.

La logique du projet est surtout :
- traiter les prix < 0.5 comme manquants
- exclure certaines firmes de l’univers si nécessaire
- laisser apparaître les vrais rendements extrêmes, notamment en cas de delisting

---

In [ ]:
# Summarize the distribution of cleaned monthly returns

returns.stack().describe()

### 5.4 Constructing the return matrix

The final return matrix is defined after cleaning the monthly return series and handling extreme observations.

This matrix constitutes the main input for the dynamic investment universe, covariance estimation, and portfolio backtesting procedures developed in the following sections.

In [ ]:
# Create the final return matrix used for portfolio construction

returns_matrix = returns.copy()

In [ ]:
# Check the dimensions of the final return matrix used for portfolio analysis
print("Return matrix shape:", returns_matrix.shape)

In [ ]:
# Inspect a small portion of the final return matrix

returns_matrix.iloc[:5, :5]

## 6. Dynamic Investment Universe

This section defines the dynamic investment universe used for portfolio construction.

Rather than assuming that all firms are investable at all times, the eligible universe is redefined each year based on data availability and basic screening rules. This ensures that the portfolio optimization is performed on firms with sufficiently informative return histories.


---

[GUIDE ÉQUIPE – À SUPPRIMER AVANT RENDU]

Ici on construit l’univers investissable annuel.

Ce n’est plus seulement le filtre régional EM :
on applique maintenant les règles de disponibilité des données.

L’idée est simple :
à chaque année de rebalancement, on regarde les 120 mois précédents
et on garde seulement les firmes qui ont un historique exploitable.

---



### 6.1 Defining the rebalancing years

Portfolio weights are re-estimated annually. For each rebalancing year, the estimation window uses the previous 120 monthly observations.

In [ ]:
# Define the years for annual portfolio rebalancing
rebalance_years = list(range(2014, 2026))

rebalance_years

### 6.2 Defining firm eligibility rules

A firm is considered eligible if it has a sufficiently long return history and does not exhibit an excessive proportion of zero returns within the estimation window.

These rules help exclude firms with limited trading activity or insufficient historical information.


---

[GUIDE ÉQUIPE – À SUPPRIMER AVANT RENDU]

On reprend ici les règles utilisées dans l’ancien notebook :

- au moins 36 observations non manquantes
- pas plus de 50% de rendements nuls

Cela permet d’éviter d’inclure des actions trop illiquides ou trop mal renseignées.

---



In [ ]:
# Determine whether an asset is eligible for portfolio construction
# based on return history, liquidity indicators, end-of-year price availability,
# and carbon data availability

def is_eligible(series, price_available, carbon_available):
    
    # Require a minimum number of historical return observations
    if series.count() < MIN_HISTORY_MONTHS:
        return False
    
    # Exclude assets with too many zero returns (potential illiquidity)
    zero_ratio = (series == 0).mean()
    if zero_ratio > STALE_RETURN_THRESHOLD:
        return False
    
    # Exclude firms whose end-of-year price is missing
    if not price_available:
        return False
    
    # Exclude firms without carbon data available at the end of year Y
    if not carbon_available:
        return False
    
    return True

### 6.3 Constructing the yearly investment universe

For each rebalancing year, the investment universe is defined using the previous 120 months of returns. Eligibility is assessed firm by firm according to the rules defined above.

In [ ]:
# Build the yearly investment universe using past return information only
# and require both price and carbon data availability at the end of year Y

universe_by_year = {}

for year in rebalance_years:
    end = pd.Timestamp(f"{year-1}-12-31")
    window = returns_matrix.loc[:, :end].iloc[:, -ROLLING_WINDOW_MONTHS:]

    eligible_assets = []

    for isin in window.index:
        series = window.loc[isin]

        cond_history = series.count() >= MIN_HISTORY_MONTHS
        cond_stale = (series == 0).mean() <= STALE_RETURN_THRESHOLD

        cond_price = False
        if isin in price_data.index:
            available_price_dates = price_data.columns[price_data.columns <= end]
            if len(available_price_dates) > 0:
                last_price_date = available_price_dates.max()
                cond_price = pd.notna(price_data.loc[isin, last_price_date])

        cond_carbon = False
        if isin in carbon_data.index and end in carbon_data.columns:
            cond_carbon = pd.notna(carbon_data.loc[isin, end])

        if cond_history and cond_stale and cond_price and cond_carbon:
            eligible_assets.append(isin)

    universe_by_year[year] = eligible_assets

### 6.4 Inspecting the dynamic universe

The size of the investment universe varies over time as firms enter or leave the sample depending on data availability and eligibility.

In [ ]:
# Quick diagnostic: size of the investable universe each year

for year in rebalance_years:
    print(f"{year}: {len(universe_by_year[year])} firms")

## 7. Minimum Variance Portfolio Construction

This section constructs the long-only minimum variance portfolio.

For each rebalancing year, the covariance matrix of asset returns is estimated using the previous 120 monthly observations. Portfolio weights are then obtained by solving a constrained quadratic optimization problem under full-investment and long-only constraints.


---

[GUIDE ÉQUIPE – À SUPPRIMER AVANT RENDU]

Cette section sert uniquement à construire les poids du portefeuille minimum variance.

On ne calcule pas encore les rendements réalisés ici.
On fait seulement :
- estimation de la matrice de covariance
- résolution du problème d’optimisation
- stockage des poids par année

Le backtest viendra dans la section 8.

---



### 7.1 Defining the optimization problem

The minimum variance portfolio is obtained by minimizing portfolio variance subject to two constraints:
- portfolio weights must sum to one,
- short-selling is not allowed.


---

[GUIDE ÉQUIPE – À SUPPRIMER AVANT RENDU]

Ici on définit mathématiquement le problème :
on cherche les poids qui minimisent la variance du portefeuille.

Contraintes :
- somme des poids = 1
- poids entre 0 et 1
=> portefeuille long-only

---


In [ ]:
def compute_mv_weights(cov_matrix):
    """
    Compute long-only minimum variance portfolio weights.
    """

    n = cov_matrix.shape[0]

    # Check that covariance matrix does not contain invalid values
    if np.isnan(cov_matrix).any():
        raise ValueError("Covariance matrix contains NaN values.")

    if np.isinf(cov_matrix).any():
        raise ValueError("Covariance matrix contains infinite values.")

    def objective(w):
        return w.T @ cov_matrix @ w

    constraints = ({
        "type": "eq",
        "fun": lambda w: np.sum(w) - 1
    })

    bounds = [(0, 1)] * n
    w0 = np.ones(n) / n

    result = minimize(
        objective,
        w0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints
    )

    if not result.success:
        raise ValueError(f"Minimum variance optimization failed: {result.message}")

    return result.x

### 7.2 Computing yearly minimum variance weights

For each rebalancing year, the covariance matrix is estimated using the firms belonging to the corresponding dynamic investment universe.

The optimization is then solved separately for each year, producing a time series of yearly portfolio weights.


---

[GUIDE ÉQUIPE – À SUPPRIMER AVANT RENDU]

Ici on applique la fonction d’optimisation année par année.

Pour chaque année :
- on prend l’univers investissable de cette année
- on prend les 120 mois précédents
- on calcule la covariance
- on obtient les poids minimum variance

---



In [ ]:
# Estimate yearly minimum variance portfolio weights using a rolling return window

mv_weights_by_year = {}

for year in rebalance_years:

    universe = universe_by_year[year]
    end = pd.Timestamp(f"{year-1}-12-31")

    # Rolling estimation window of monthly returns
    window = returns_matrix.loc[universe, :end].iloc[:, -ROLLING_WINDOW_MONTHS:]

    # Covariance matrix of monthly returns
    cov = window.T.cov()

    # Compute minimum variance weights
    weights = compute_mv_weights(cov.values)

    # Store weights as a pandas Series
    mv_weights_by_year[year] = pd.Series(weights, index=cov.index)

In [ ]:
# Quick diagnostic of the minimum variance portfolio for the first rebalance year

print("Number of firms in 2014 MV portfolio:", len(mv_weights_by_year[2014]))
print("Sum of weights in 2014:", mv_weights_by_year[2014].sum())

mv_weights_by_year[2014].head()

## 8. Backtesting the Minimum Variance Portfolio




This section evaluates the out-of-sample performance of the minimum variance portfolio.

For each rebalancing year, the portfolio weights estimated in the previous section are applied to the realized monthly returns of the corresponding year. Between rebalancing dates, portfolio weights evolve dynamically as asset returns accumulate.


---
[GUIDE ÉQUIPE – À SUPPRIMER AVANT RENDU]

Important : version finale retenue pour la Part I.

Le projet demande explicitement que les poids du portefeuille minimum variance évoluent chaque mois à l’intérieur de l’année, même si la décision d’investissement est prise une fois par an.

Donc :
- estimation des poids en début d'année
- pas de ré-optimisation mensuelle
- mais mise à jour dynamique des poids mois par mois

---



### 8.1 Defining the backtesting procedure

The backtesting procedure computes monthly portfolio returns over each calendar year using the weights estimated at the beginning of that year.

Between rebalancing dates, weights are updated dynamically to reflect the cumulative realized performance of the underlying assets.


---


[GUIDE ÉQUIPE – À SUPPRIMER AVANT RENDU]

Ici on applique exactement la formule du projet :

si une action monte pendant le mois, son poids augmente mécaniquement dans le portefeuille au mois suivant.

C'est donc un portefeuille buy-and-hold à l'intérieur de l'année, avec rebalancement annuel.

---



In [ ]:
def backtest_dynamic_portfolio(returns_matrix, weights, year):
    """
    Compute monthly portfolio returns over a given calendar year.

    The portfolio starts the year with weights estimated at the end of the
    previous year. Between rebalancing dates, weights evolve dynamically
    according to realized asset returns.
    """

    returns_year = returns_matrix.loc[weights.index, f"{year}-01-01":f"{year}-12-31"]

    current_weights = weights.copy()
    portfolio_returns = []

    for date in returns_year.columns:
        month_returns = returns_year[date]

        # Keep only assets with available returns for the month
        valid_mask = month_returns.notna()
        month_returns = month_returns[valid_mask]
        current_weights = current_weights[valid_mask]

        # Renormalize weights before computing portfolio return
        current_weights = current_weights / current_weights.sum()

        portfolio_return = (current_weights * month_returns).sum()
        portfolio_returns.append(portfolio_return)

        # Update weights dynamically within the year
        current_weights = current_weights * (1 + month_returns)
        current_weights = current_weights / current_weights.sum()

    portfolio_returns = pd.Series(
        portfolio_returns,
        index=returns_year.columns,
        name=f"mv_returns_{year}"
    )

    return portfolio_returns

### 8.2 Running the yearly minimum variance backtest

The yearly backtest is performed by combining the estimated minimum variance weights with the realized monthly returns of the corresponding year.

The resulting yearly return series are then concatenated into a single out-of-sample series covering the 2014–2025 period.


---

[GUIDE ÉQUIPE – À SUPPRIMER AVANT RENDU]

On applique la fonction de backtest dynamique année par année.

Pour chaque année :
- on prend les poids minimum variance calculés en section 7
- on calcule les rendements mensuels réalisés
- on concatène ensuite toutes les années
---



In [ ]:
# Backtest the minimum variance portfolio year by year and build the full out-of-sample return series

mv_returns_by_year = {}

for year in rebalance_years:
    weights = mv_weights_by_year[year]
    realized_returns = backtest_dynamic_portfolio(returns_matrix, weights, year)
    mv_returns_by_year[year] = realized_returns

mv_returns_oos = pd.concat(mv_returns_by_year.values()).sort_index()

In [ ]:
# Validate the out-of-sample minimum variance return series and compute total cumulative return

print("Out-of-sample MV return series shape:", mv_returns_oos.shape)
print("First date:", mv_returns_oos.index.min())
print("Last date:", mv_returns_oos.index.max())

mv_total_cum_return = (1 + mv_returns_oos).prod() - 1
print("Minimum Variance total cumulative return:", mv_total_cum_return)

## 9. Value-Weighted Benchmark Portfolio

This section constructs the value-weighted benchmark portfolio used for comparison with the minimum variance strategy.

To ensure consistency, the benchmark is built on the same dynamic investment universe as the minimum variance portfolio. Within each year, monthly benchmark returns are computed using lagged market capitalization weights: the market capitalization observed at the end of month \(t\) is used to weight stock returns in month \(t+1\).


---


[GUIDE ÉQUIPE – À SUPPRIMER AVANT RENDU]

Ici on construit le benchmark du projet.

Même logique que le portefeuille minimum variance :
- même univers dynamique
- mêmes années de rebalancement

Mais les poids viennent simplement des market caps.

Ce portefeuille sert de point de comparaison pour évaluer la performance du minimum variance.

---



### 9.1 Computing yearly value-weighted portfolio weights

For each rebalancing year, the benchmark is evaluated on the corresponding dynamic investment universe. Monthly portfolio weights are computed from firms' market capitalizations observed at the end of the previous month. These lagged value weights are then applied to the realized monthly returns of the current month.


---


[GUIDE ÉQUIPE – À SUPPRIMER AVANT RENDU]

Important :
dans la version finale, le benchmark value-weighted est construit mois par mois.

Ce choix est volontaire :
- c'est la logique de l'ancien notebook
- c'est ce qui explique le benchmark plus performant observé auparavant

Donc ici on ne garde pas des poids annuels fixes.

---



In [ ]:
def backtest_value_weighted_portfolio(returns_matrix, market_caps_data, universe, year):
    """
    Compute monthly returns of the value-weighted benchmark portfolio
    using lagged market capitalization weights.
    """

    returns_year = returns_matrix.loc[universe, f"{year}-01-01":f"{year}-12-31"]

    portfolio_returns = []

    for date in returns_year.columns:
        previous_dates = market_caps_data.columns[market_caps_data.columns < date]

        if len(previous_dates) == 0:
            portfolio_returns.append(np.nan)
            continue

        prev_date = previous_dates.max()

        caps = market_caps_data.loc[universe, prev_date]
        caps = pd.to_numeric(caps, errors="coerce")

        month_returns = returns_matrix.loc[universe, date]

        # Keep only assets with both market cap and return available
        valid_mask = caps.notna() & month_returns.notna()
        caps = caps[valid_mask]
        month_returns = month_returns[valid_mask]

        if caps.empty or caps.sum() <= 0:
            portfolio_returns.append(np.nan)
            continue

        weights = caps / caps.sum()
        portfolio_return = (weights * month_returns).sum()

        portfolio_returns.append(portfolio_return)

    portfolio_returns = pd.Series(
        portfolio_returns,
        index=returns_year.columns,
        name=f"vw_returns_{year}"
    )

    return portfolio_returns

In [ ]:
# Check the structure of the monthly value-weighted benchmark weights
# and verify that weights sum to one

print("VW monthly weights shape:", vw_weights_monthly.shape)
print("Sum of weights for first month:", vw_weights_monthly.iloc[:, 0].sum())
print("Sum of weights for last month:", vw_weights_monthly.iloc[:, -1].sum())

### 9.2 Backtesting the value-weighted benchmark

The out-of-sample benchmark return series is constructed year by year using the same dynamic investment universe as the minimum variance strategy. For each month, the benchmark return is obtained as the weighted average of realized stock returns, where the weights are based on lagged end-of-month market capitalizations.


---

[GUIDE ÉQUIPE – À SUPPRIMER AVANT RENDU]

Ici on calcule directement le benchmark mensuel :

rendement du mois = somme(returns mensuels × poids mensuels de market cap)

C'est cette logique qui était utilisée dans l'ancien notebook.

---



In [ ]:
# Backtest the value-weighted benchmark year by year

vw_returns_by_year = {}

for year in rebalance_years:
    universe = universe_by_year[year]
    realized_returns = backtest_value_weighted_portfolio(
        returns_matrix,
        market_caps_data,
        universe,
        year
    )
    vw_returns_by_year[year] = realized_returns

vw_returns_oos = pd.concat(vw_returns_by_year.values()).sort_index()

In [ ]:
# Validate the out-of-sample value-weighted return series and compute total cumulative return

print("Out-of-sample VW return series shape:", vw_returns_oos.shape)
print("First date:", vw_returns_oos.index.min())
print("Last date:", vw_returns_oos.index.max())

vw_total_cum_return = (1 + vw_returns_oos).prod() - 1
print("Value-Weighted total cumulative return:", vw_total_cum_return)

## 10. Performance Comparison

This section compares the out-of-sample performance of the minimum variance portfolio and the value-weighted benchmark.

We compute several summary statistics using monthly returns over the 2014–2025 period, including:

- annualized average return
- annualized volatility
- Sharpe ratio, computed with the monthly risk-free rate
- minimum monthly return
- maximum monthly return
- total cumulative return

These indicators allow us to assess both the profitability and the risk profile of the two strategies.


---

[GUIDE ÉQUIPE – À SUPPRIMER AVANT RENDU]

On compare ici les deux stratégies :

- Minimum Variance
- Value Weighted (benchmark)

On utilise les rendements mensuels entre 2014 et 2025 pour calculer :

- rendement annualisé
- volatilité annualisée
- Sharpe ratio, computed with the monthly risk-free rate
- rendement mensuel minimum
- rendement mensuel maximum
- rendement cumulé total

---



### 10.1 Performance statistics

Performance statistics are computed from the monthly return series obtained during the backtesting period.

In [ ]:
# Compare the out-of-sample performance of the two portfolios
# Sharpe ratio uses excess returns: annualized(Rp - Rf) / annualized volatility.

from src.reporting import build_performance_table, prepare_risk_free_rate

risk_free_raw = pd.read_excel(DATA_RAW / "Risk_Free_Rate_2025.xlsx")
risk_free_rate = prepare_risk_free_rate(risk_free_raw)

performance = build_performance_table(
    mv_returns_oos=mv_returns_oos,
    vw_returns_oos=vw_returns_oos,
    risk_free_rate=risk_free_rate,
)

performance

### 10.2 Cumulative return comparison

To visualize the evolution of portfolio performance over time, cumulative returns are computed from the monthly return series of both strategies.


---

[GUIDE ÉQUIPE – À SUPPRIMER AVANT RENDU]

On transforme les rendements mensuels en valeur cumulée :

(1 + returns).cumprod()

Cela permet de voir comment 1$ investi évoluerait dans chaque portefeuille.

---



In [ ]:
# Compute cumulative wealth indices for both strategies

mv_cumulative = (1 + mv_returns_oos).cumprod()
vw_cumulative = (1 + vw_returns_oos).cumprod()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(mv_cumulative, label="Minimum Variance", linewidth=2)
plt.plot(vw_cumulative, label="Value Weighted", linewidth=2)

plt.title("Cumulative Portfolio Performance (2014–2025)")
plt.ylabel("Growth of $1 Investment")
plt.xlabel("Date")

plt.legend()
plt.grid()

# Save figure before displaying it
plt.savefig(FIGURES_DIR / "cumulative_portfolio_performance.png", dpi=300, bbox_inches="tight")

plt.show()

## 11. Part I Summary


In this first part of the project, we constructed and evaluated a standard minimum variance portfolio using the Emerging Markets universe assigned to our group.

After cleaning the Datastream dataset, we computed monthly returns and defined a dynamic investment universe based on data availability and eligibility criteria. Using a rolling window of 120 monthly observations, we estimated the covariance matrix and constructed a long-only minimum variance portfolio, rebalanced annually.

The out-of-sample performance of this portfolio over the 2014–2025 period was then compared with that of the value-weighted benchmark portfolio. The empirical results show that the benchmark achieved a higher cumulative return over the sample period, while the minimum variance portfolio provides an alternative allocation rule based on variance minimization.

This first part establishes the empirical framework that will be used in the remainder of the project. In Part II, we extend this framework by incorporating carbon-related constraints into the portfolio construction process.


---

[GUIDE ÉQUIPE – À SUPPRIMER AVANT RENDU]

Cette section doit rester courte.

Objectifs :
- rappeler ce que l’on a fait dans la Part I
- mentionner brièvement le résultat principal
- introduire la Part II (contrainte carbone)

L’analyse détaillée et la discussion seront faites dans le rapport PDF.

---



# 12. Exports

In [ ]:
# Save key outputs of Part I for reproducibility and later analysis

mv_returns_oos.to_csv(TABLES_DIR / "mv_returns_oos.csv")
vw_returns_oos.to_csv(TABLES_DIR / "vw_returns_oos.csv")
performance.to_csv(TABLES_DIR / "portfolio_performance_summary.csv", index=False)

print("Outputs successfully saved in:", TABLES_DIR)

In [ ]:
# =========================================================
# Export results to the official Part I Excel template
# =========================================================

template_path = BASE_DIR / "Template for Part I-SAAM.xlsx"
output_template = OUTPUTS / "Part_I_results_filled.xlsx"

with pd.ExcelWriter(output_template, engine="openpyxl", mode="w") as writer:
    
    # Sheet 1 — Portfolio performance summary
    performance.to_excel(writer, sheet_name="Portfolio Performance", index=False)

    # Sheet 2 — Monthly returns
    returns_table = pd.DataFrame({
        "Date": mv_returns_oos.index,
        "Minimum Variance": mv_returns_oos.values,
        "Value Weighted": vw_returns_oos.values
    })
    
    returns_table.to_excel(writer, sheet_name="Monthly Returns", index=False)

print("Template successfully filled and saved to:")
print(output_template)